In [1]:
!pip install httpx beautifulsoup4 lxml

In [13]:
import httpx
from bs4 import BeautifulSoup
import json
import re

def test_cricbuzz_scraper(match_id):
    url = f"https://www.cricbuzz.com/live-cricket-scores/{match_id}/match-slug"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"--- Fetching Match: {match_id} ---")

    with httpx.Client(headers=headers, timeout=10) as client:
        response = client.get(url)

    if response.status_code != 200:
        print(f"Failed to fetch! Status Code: {response.status_code}")
        return

    soup = BeautifulSoup(response.text, 'lxml')

    # --- PART 1: Basic Scrape (Visible HTML) ---
    try:
        status = soup.find("div", class_="text-cbTextLink").text.strip()
        scores = [div.text.strip() for div in soup.find_all("div", class_="text-lg font-bold")]
    except AttributeError:
        status = "Live Match / Not Finished"
        scores = []

    # --- PART 2: Extracting Ball-by-Ball Commentary ---
    commentary = []
    # Find containers that have ball numbers (e.g., '18.3')
    ball_containers = soup.find_all("div", class_=re.compile("font-bold text-center"))

    for container in ball_containers[:5]: # Get last 5 balls
        ball_num = container.text.strip()
        # The text is in the parent's next sibling div
        try:
            parent = container.find_parent()
            comm_text = parent.find_next_sibling("div").text.strip()
            commentary.append(f"{ball_num}: {comm_text}")
        except:
            continue

    # --- DISPLAY RESULTS ---
    print(f"MATCH STATUS: {status}")
    print(f"SCOREBOARD: {' vs '.join(scores)}")
    print("\nRECENT COMMENTARY:")
    for c in commentary:
        print(c)

# Run the test with the Match ID from your HTML
test_cricbuzz_scraper("151913")

--- Fetching Match: 151913 ---
MATCH STATUS: Have Your Say
SCOREBOARD: 

RECENT COMMENTARY:
0-0(0.2): 
0.2: Mohammed Siraj to Sanju Samson, no run
0.1: Mohammed Siraj to Sanju Samson, no run, beaten! 132.8kph, beautiful shape for Siraj. On a length and significant movement off the deck. Samson stays leg-side
: Sanju Samson and Ruturaj Gaikwad are at the crease. Sanju Samson is on strike. Mohammed Siraj will open the attack
: Interestingly, GT have left out Prasidh Krishna for this encounter. He's been their leading wicket-taker this season with 12 wickets in 7 matches. CSK, meanwhile have brought in Urvil Patel while Sarfaraz Khan and Mukesh Choudhary are in the Impact subs list. Shubman Gill doing all the talking in the team huddle before they disperse. In walk the CSK openers. Both of them have a good record against GT. Siraj to start the proceedings for the visitors. One slip in place and here we go...


In [3]:
import httpx
from bs4 import BeautifulSoup
import re

def get_all_match_ids():
    url = "https://www.cricbuzz.com/cricket-match/live-scores"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = httpx.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'lxml')

    matches = []
    # Cricbuzz match links follow the pattern: /live-cricket-scores/ID/match-slug
    links = soup.find_all('a', href=re.compile(r'/live-cricket-scores/\d+/'))

    for link in links:
        href = link['href']
        match_id = href.split('/')[2]
        title = link.get('title', 'Unknown Match')

        # Avoid duplicates
        if not any(m['id'] == match_id for m in matches):
            matches.append({"id": match_id, "title": title})

    return matches

# Test finding IDs
all_matches = get_all_match_ids()
for m in all_matches:
    print(f"ID: {m['id']} | Match: {m['title']}")

ID: 151913 | Match: Chennai Super Kings vs Gujarat Titans, 37th Match - GT opt to bowl 
ID: 149347 | Match: Hyderabad Kingsmen vs Rawalpindiz, 39th Match - RWP opt to bowl 
ID: 151924 | Match: Lucknow Super Giants vs Kolkata Knight Riders, 38th Match - Preview 
ID: 151902 | Match: Rajasthan Royals vs Sunrisers Hyderabad, 36th Match - SRH won 
ID: 144419 | Match: India Women vs South Africa Women, 4th T20I - INDW won 
ID: 150118 | Match: Bangladesh vs New Zealand, 1st T20I - Preview 
ID: 153160 | Match: Nepal vs United Arab Emirates, 98th Match - NEP Won 
ID: 153171 | Match: United Arab Emirates vs Oman, 99th Match - Preview 
ID: 151891 | Match: Delhi Capitals vs Punjab Kings, 35th Match - Complete 
ID: 149325 | Match: Quetta Gladiators vs Karachi Kings, 37th Match - KRK Won 
ID: 149336 | Match: Peshawar Zalmi vs Lahore Qalandars, 38th Match - Complete 
ID: 149358 | Match: Islamabad United vs Multan Sultans, 40th Match - Preview 
ID: 142143 | Match: Warwickshire vs Nottinghamshire, 13th

In [19]:
import httpx
import asyncio
import json

async def get_raw_combined_json(match_id):
    # Base configuration from your shared curls
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0",
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": f"https://www.cricbuzz.com/live-cricket-scores/{match_id}",
        "Content-Type": "application/json",
        "Cookie": "cbgeo=IN; cbzads=IN|not_set|KA|BANGALORE",
        "Connection": "keep-alive"
    }

    # The 3 endpoints you discovered
    endpoints = {
        "commentary": f"https://www.cricbuzz.com/api/mcenter/comm/{match_id}",
        "livescore": f"https://www.cricbuzz.com/api/mcenter/livescore/{match_id}",
        "views": f"https://www.cricbuzz.com/api/mcenter/views-counter/{match_id}"
    }

    async with httpx.AsyncClient(headers=headers, http2=True, timeout=10) as client:
        # Create concurrent tasks
        tasks = {key: client.get(url) for key, url in endpoints.items()}
        responses = await asyncio.gather(*tasks.values())

        # Zip results back to keys
        results = dict(zip(tasks.keys(), [r.json() if r.status_code == 200 else {"error": r.status_code} for r in responses]))

        # Final Master Object
        master_json = {
            "matchId": match_id,
            "data": results
        }

        return master_json

# --- Execution for Colab ---
# Since Colab has an active event loop, we use this helper:
import nest_asyncio
nest_asyncio.apply()

match_id = "151913"
raw_output = asyncio.run(get_raw_combined_json(match_id))

# Print beautiful Raw JSON
print(json.dumps(raw_output, indent=4))

{
    "matchId": "151913",
    "data": {
        "commentary": {
            "matchCommentary": {
                "1777205348699": {
                    "matchId": 151913,
                    "commType": "commentary",
                    "commText": "Akeal Hosein to Sai Sudharsan, 1 run, to square leg",
                    "inningsId": 2,
                    "event": [
                        "none",
                        "all"
                    ],
                    "ballMetric": 0.1,
                    "teamName": "GT",
                    "timestamp": 1777205348699,
                    "overSeparator": null,
                    "batsmanDetails": {
                        "playerId": 13866,
                        "playerName": "Sai Sudharsan"
                    },
                    "bowlerDetails": {
                        "playerId": 8435,
                        "playerName": "Akeal Hosein"
                    }
                },
                "1777205301072": {
     

In [18]:
import httpx
import asyncio
import json
import re
from bs4 import BeautifulSoup

# Required for Google Colab to handle nested event loops
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

async def get_match_raw_data(client, match_id, match_title):
    """Fetches data from all 3 endpoints concurrently for a specific match_id."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0",
        "Accept": "application/json",
        "Referer": f"https://www.cricbuzz.com/live-cricket-scores/{match_id}"
    }

    endpoints = {
        "livescore": f"https://www.cricbuzz.com/api/mcenter/livescore/{match_id}",
        "commentary": f"https://www.cricbuzz.com/api/mcenter/comm/{match_id}",
        "views": f"https://www.cricbuzz.com/api/mcenter/views-counter/{match_id}"
    }

    match_results = {"matchId": match_id, "title": match_title, "endpoints": {}}

    # Fetch all 3 endpoints for this match concurrently
    tasks = [client.get(url, headers=headers) for url in endpoints.values()]
    responses = await asyncio.gather(*tasks)

    for key, response in zip(endpoints.keys(), responses):
        if response.status_code == 200:
            match_results["endpoints"][key] = response.json()
        else:
            match_results["endpoints"][key] = {"error": f"HTTP {response.status_code}"}

    return match_results

async def scrape_all_live_matches():
    """First finds all match-ids, then gets all data for each."""
    index_url = "https://www.cricbuzz.com/cricket-match/live-scores"
    headers = {"User-Agent": "Mozilla/5.0"}

    async with httpx.AsyncClient(http2=True, timeout=10) as client:
        # 1. Discover all Match IDs
        response = await client.get(index_url, headers=headers)
        soup = BeautifulSoup(response.text, 'lxml')

        matches_to_fetch = []
        links = soup.find_all('a', href=re.compile(r'/live-cricket-scores/\d+/'))

        for link in links:
            m_id = link['href'].split('/')[2]
            title = link.get('title', 'Unknown Match')
            if not any(m['id'] == m_id for m in matches_to_fetch):
                matches_to_fetch.append({"id": m_id, "title": title})

        # 2. Concurrently fetch data for ALL discovered matches
        print(f"Found {len(matches_to_fetch)} matches. Fetching details...\n")

        master_tasks = [get_match_raw_data(client, m['id'], m['title']) for m in matches_to_fetch]
        all_match_data = await asyncio.gather(*master_tasks)

        return all_match_data

# Execute and Print
if __name__ == "__main__":
    final_output = asyncio.run(scrape_all_live_matches())

    # Print the full raw JSON for the first match found as a sample
    if final_output:
        print(json.dumps(final_output, indent=4))

Streaming output truncated to the last 5000 lines.
                        },
                        "bowlerDetails": {
                            "playerId": 49282,
                            "playerName": "Ed Middleton"
                        }
                    },
                    "1777137356244": {
                        "matchId": 142838,
                        "commType": "commentary",
                        "commText": "Buuren to Andersson, no run",
                        "inningsId": 2,
                        "event": [
                            "over-break",
                            "maiden_over",
                            "all"
                        ],
                        "ballMetric": 39.6,
                        "teamName": "DERBY",
                        "timestamp": 1777137356244,
                        "overSeparator": {
                            "timestamp": 1777137356244,
                            "overNumber": 40,
                    